In [0]:
%sql
CREATE TABLE IF NOT EXISTS customer360_dev.control.watermark
(
    entity_name           STRING,
    source_system         STRING,
    last_processed_value  TIMESTAMP,
    last_processed_file   STRING,
    last_run_id           STRING,
    load_status           STRING,
    updated_at            TIMESTAMP
)
USING DELTA;

In [0]:
%sql
MERGE INTO customer360_dev.control.watermark AS tgt
USING (
    SELECT * FROM VALUES
        ('customers',       'CRM'),
        ('addresses',       'CRM'),
        ('subscriptions',   'SUBSCRIPTION_PLATFORM'),
        ('products',        'PRODUCT_CATALOG'),
        ('orders',          'ORDER_MANAGEMENT'),
        ('order_items',     'ORDER_MANAGEMENT'),
        ('payments',        'PAYMENT_PLATFORM'),
        ('support_tickets', 'SUPPORT_PLATFORM')
    AS v(entity_name, source_system)
) AS src
ON tgt.entity_name = src.entity_name

WHEN NOT MATCHED THEN
INSERT (
    entity_name,
    source_system,
    last_processed_value,
    last_processed_file,
    last_run_id,
    load_status,
    updated_at
)
VALUES (
    src.entity_name,
    src.source_system,
    NULL,
    NULL,
    NULL,
    'INITIAL_BASELINE_COMPLETE',
    current_timestamp()
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
8,0,0,8


Incremental files arrive

        ↓

Check watermark

        ↓

Read only new changes

        ↓

Validate / deduplicate

        ↓

Apply CDC logic

        ↓

Reconcile

        ↓
        
Update watermark only after SUCCESS

In [0]:
%sql
SELECT *
FROM customer360_dev.control.watermark
ORDER BY entity_name;

entity_name,source_system,last_processed_value,last_processed_file,last_run_id,load_status,updated_at
addresses,CRM,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
customers,CRM,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
order_items,ORDER_MANAGEMENT,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
orders,ORDER_MANAGEMENT,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
payments,PAYMENT_PLATFORM,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
products,PRODUCT_CATALOG,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
subscriptions,SUBSCRIPTION_PLATFORM,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
support_tickets,SUPPORT_PLATFORM,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
